In [1]:
import numpy as np
import pandas as pd
import faiss
import json
from pathlib import Path
from tqdm.auto import tqdm

ARTIFACT_DIR = Path(r"D:\PixoAnalytics\Image Retrieval Artifacts")
EMBEDDING_DIR = ARTIFACT_DIR / "embeddings"

EMBEDDINGS_PATH = EMBEDDING_DIR / "embeddings.npy"
IMAGE_IDS_PATH = EMBEDDING_DIR / "image_ids.npy"

INDEX_PATH = EMBEDDING_DIR / "faiss_index.bin"
VALID_POSITIONS_PATH = EMBEDDING_DIR / "valid_positions.npy"

# Load using memory mapping to avoid unnecessary RAM usage
embeddings = np.load(EMBEDDINGS_PATH, mmap_mode="r")
image_ids = np.load(IMAGE_IDS_PATH, allow_pickle=False)

print("Embeddings:", embeddings.shape)
print("Image IDs:", image_ids.shape)
print("FAISS version:", faiss.__version__)

assert embeddings.shape == (398170, 512)
assert len(image_ids) == len(embeddings)

Embeddings: (398170, 512)
Image IDs: (398170,)
FAISS version: 1.16.0


In [2]:
valid_positions = []

CHUNK_SIZE = 10000

for start in tqdm(range(0, len(embeddings), CHUNK_SIZE)):

    end = min(start + CHUNK_SIZE, len(embeddings))

    batch = np.asarray(embeddings[start:end])

    norms = np.linalg.norm(batch, axis=1)

    valid_mask = (
        np.isfinite(batch).all(axis=1)
        & np.isfinite(norms)
        & (norms > 1e-6)
    )

    positions = np.arange(start, end)[valid_mask]

    valid_positions.extend(positions.tolist())

valid_positions = np.asarray(valid_positions, dtype=np.int64)

print("Total embeddings:", len(embeddings))
print("Valid embeddings:", len(valid_positions))
print("Invalid embeddings:", len(embeddings) - len(valid_positions))

  0%|          | 0/40 [00:00<?, ?it/s]

Total embeddings: 398170
Valid embeddings: 398170
Invalid embeddings: 0


In [3]:
EMBEDDING_DIM = 512
INDEX_BATCH_SIZE = 10000

# Exact inner-product search
index = faiss.IndexFlatIP(EMBEDDING_DIM)

for start in tqdm(
    range(0, len(valid_positions), INDEX_BATCH_SIZE),
    desc="Building FAISS index"
):

    positions = valid_positions[start:start + INDEX_BATCH_SIZE]

    batch = np.asarray(embeddings[positions], dtype=np.float32)

    # Normalize again to guarantee cosine similarity
    norms = np.linalg.norm(batch, axis=1, keepdims=True)
    batch = batch / np.maximum(norms, 1e-12)

    # FAISS expects contiguous float32 arrays
    batch = np.ascontiguousarray(batch)

    index.add(batch)

# Save the mapping from FAISS positions to original embedding positions
np.save(VALID_POSITIONS_PATH, valid_positions)

# Save the FAISS index
faiss.write_index(index, str(INDEX_PATH))

print("FAISS index built successfully!")
print("Indexed vectors:", index.ntotal)
print("Index dimension:", index.d)
print("Index file size:",
      round(INDEX_PATH.stat().st_size / 1024**2, 2), "MB")

Building FAISS index:   0%|          | 0/40 [00:00<?, ?it/s]

FAISS index built successfully!
Indexed vectors: 398170
Index dimension: 512
Index file size: 777.68 MB


In [5]:
# Choose one valid image
query_position = int(valid_positions[0])

# Create an independent, writable copy
query_vector = np.array(
    embeddings[query_position:query_position + 1],
    dtype=np.float32,
    copy=True
)

# Normalize query
query_vector /= np.linalg.norm(
    query_vector,
    axis=1,
    keepdims=True
)

# Search top 5
distances, indices = index.search(
    np.ascontiguousarray(query_vector),
    5
)

# Convert FAISS positions to original embedding positions
original_positions = valid_positions[indices[0]]

retrieved_ids = image_ids[original_positions]

print("Query image ID:", image_ids[query_position])
print("\nRetrieved results:")

for rank, (image_id, score) in enumerate(
    zip(retrieved_ids, distances[0]), start=1
):
    print(f"{rank}. {image_id} | Similarity: {score:.4f}")

Query image ID: 81iZlv3bjpL

Retrieved results:
1. 81iZlv3bjpL | Similarity: 1.0000
2. 91603-q1vCL | Similarity: 0.9316
3. 91eqBkW06wL | Similarity: 0.9306
4. 81FXnJClGZL | Similarity: 0.9302
5. 81RHmO-snwL | Similarity: 0.9286
